In [ ]:
import glob
import os

import h5py
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from keras.callbacks import EarlyStopping
from keras.models import Sequential
from keras.models import load_model
from keras.optimizers import Adam
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from tensorflow.keras.layers import (
    Input, Flatten, Dense, Conv2D, Dropout
)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

best_params = None

In [ ]:
IMPACT_LOCATIONS = [
    'Back',
    'Back Left',
    'Back Neck',
    'Back Right',
    'Back Top Left',
    'Back Top Right',
    'Bottom Back',
    'Bottom Back Left',
    'Bottom Back Right',
    'Bottom Front',
    'Bottom Left',
    'Bottom Right',
    'Front',
    'Front Bottom Left',
    'Front Bottom Right',
    'Front Left',
    'Front Neck',
    'Front Right',
    'Front Top Left',
    'Front Top Right',
    'Left',
    'Left Neck',
    'Right',
    'Right Neck',
    'Top Back',
    'Top Front',
    'Top Left',
    'Top Right',
    'Unknown'
]

unknown_idx = IMPACT_LOCATIONS.index('Unknown')

data_folder = '/scratch/projects/MADS2025-CiJR/rugby_brain_strain_CNN/data'

# Make sure the path exists
if not os.path.exists(data_folder):
    print("Folder does not exist:", data_folder)
else:
    # Recursive search for all .h5 files
    file_list = glob.glob(os.path.join(data_folder, '**', '*.h5'), recursive=True)
    print(f"Found {len(file_list)} files:")
    for f in file_list:
        print(f)

X = []
y = []

for file_path in file_list:
    try:
        with h5py.File(file_path, 'r') as hf:
            for key in hf.keys():
                try:
                    group = hf[key]
                except (KeyError, OSError) as e:
                    print(f"Skipping group {key} in {file_path}: {e}")
                    continue

                if 'impact_location' not in group.attrs:
                    continue
                impact_loc = group.attrs['impact_location']

                if impact_loc[unknown_idx] == 1:
                    continue

                perm_name = 'perm_xyz'
                lin_name = 'lin_perm_xyz'
                if perm_name not in group or lin_name not in group:
                    continue

                rot = group[perm_name][:]
                lin = group[lin_name][:]

                if rot.shape != lin.shape or rot.ndim != 3 or rot.shape[0] != 1:
                    continue

                # Convert (1, 3, L) -> (3, L, 1) and stack as channels
                rot = rot.transpose(1, 2, 0)
                lin = lin.transpose(1, 2, 0)
                sample = np.concatenate([rot, lin], axis=2)  # (3, L, 2)

                X.append(sample)
                y.append(impact_loc)
    except (OSError, KeyError) as e:
        print(f"Skipping file {file_path}: {e}")
        continue

# Convert to numpy arrays
if len(X) > 0:
    X = np.stack(X, axis=0)
    y = np.stack(y, axis=0)
    y = np.delete(y, unknown_idx, axis=1)
else:
    X = np.array([])
    y = np.array([])

# 80:20 train-test split
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True,
        stratify=np.argmax(y, axis=1)
    )

    input_shape = X_train.shape[1:]
    num_classes = y_train.shape[1]

    print(f"Total samples: {len(X)}")
    print(f"Train samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")

else:
    X_train, X_test = [], []
    y_train, y_test = [], []
    input_shape = (3, 1000, 2)
    num_classes = 28

In [ ]:
# Alternative loader (rotational only, no lin_perm_)
# Run this cell instead of the main loader if you want perm_* only.

IMPACT_LOCATIONS = [
    'Back',
    'Back Left',
    'Back Neck',
    'Back Right',
    'Back Top Left',
    'Back Top Right',
    'Bottom Back',
    'Bottom Back Left',
    'Bottom Back Right',
    'Bottom Front',
    'Bottom Left',
    'Bottom Right',
    'Front',
    'Front Bottom Left',
    'Front Bottom Right',
    'Front Left',
    'Front Neck',
    'Front Right',
    'Front Top Left',
    'Front Top Right',
    'Left',
    'Left Neck',
    'Right',
    'Right Neck',
    'Top Back',
    'Top Front',
    'Top Left',
    'Top Right',
    'Unknown'
]

unknown_idx = IMPACT_LOCATIONS.index('Unknown')

data_folder = '/scratch/projects/MADS2025-CiJR/rugby_brain_strain_CNN/data'

if not os.path.exists(data_folder):
    print("Folder does not exist:", data_folder)
else:
    file_list = glob.glob(os.path.join(data_folder, '**', '*.h5'), recursive=True)
    print(f"Found {len(file_list)} files:")
    for f in file_list:
        print(f)

X = []
y = []

for file_path in file_list:
    try:
        with h5py.File(file_path, 'r') as hf:
            for key in hf.keys():
                try:
                    group = hf[key]
                except (KeyError, OSError) as e:
                    print(f"Skipping group {key} in {file_path}: {e}")
                    continue

                if 'impact_location' not in group.attrs:
                    continue
                impact_loc = group.attrs['impact_location']

                if impact_loc[unknown_idx] == 1:
                    continue

                dset_names = list(group.keys())
                perm_names = [n for n in dset_names if n.startswith('perm_') and not n.startswith('lin_perm_')]

                for perm_name in perm_names:
                    rot = group[perm_name][:]

                    if rot.ndim != 3 or rot.shape[0] != 1:
                        continue

                    # Convert (1, 3, L) -> (3, L, 1)
                    rot = rot.transpose(1, 2, 0)

                    X.append(rot)
                    y.append(impact_loc)
    except (OSError, KeyError) as e:
        print(f"Skipping file {file_path}: {e}")
        continue

if len(X) > 0:
    X = np.stack(X, axis=0)
    y = np.stack(y, axis=0)
    y = np.delete(y, unknown_idx, axis=1)
else:
    X = np.array([])
    y = np.array([])

if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        shuffle=True,
        stratify=np.argmax(y, axis=1)
    )

    input_shape = X_train.shape[1:]
    num_classes = y_train.shape[1]

    print(f"Total samples: {len(X)}")
    print(f"Train samples: {len(X_train)}")
    print(f"Test samples: {len(X_test)}")

    print(f"X_train shape: {X_train.shape}")
    print(f"y_train shape: {y_train.shape}")
    print(f"X_test shape: {X_test.shape}")
    print(f"y_test shape: {y_test.shape}")

else:
    X_train, X_test = [], []
    y_train, y_test = [], []
    input_shape = (3, 1000, 1)
    num_classes = 28

best_params = None



In [ ]:
def build_model(parameters, shape, classes):
    model = Sequential([
        Input(shape=shape),
        Conv2D(
            filters=parameters["filters"],
            kernel_size=parameters["kernel1"],
            strides=parameters["stride1"],
            activation='relu',
            padding='valid'
        ),
        Conv2D(
            filters=parameters["filters"],
            kernel_size=parameters["kernel2"],
            strides=parameters["stride2"],
            activation='relu',
            padding='valid'
        ),
        Conv2D(
            filters=parameters["filters"],
            kernel_size=parameters["kernel3"],
            strides=parameters["stride3"],
            activation='relu',
            padding='valid'
        ),
        Dropout(parameters.get("dropout", 0.2)),
        Flatten(),
        Dense(units=parameters["dense_units"], activation='relu'),
        Dense(units=classes, activation='softmax')
    ])

    optimizer = Adam(learning_rate=parameters.get("lr", 1e-4))
    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy',
        metrics=[
            tf.keras.metrics.TopKCategoricalAccuracy(k=1, name="top1_acc"),
            tf.keras.metrics.TopKCategoricalAccuracy(k=3, name="top3_acc"),
        ]
    )
    return model

In [ ]:
param_grid = [
    {
        "filters": 16,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
        "dense_units": 64,
    },
    {
        "filters": 32,
        "kernel1": (3, 10),
        "stride1": (1, 2),
        "kernel2": (1, 10),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
        "dense_units": 64,
    },
    {
        "filters": 32,
        "kernel1": (3, 7),
        "stride1": (1, 2),
        "kernel2": (1, 7),
        "stride2": (1, 2),
        "kernel3": (1, 5),
        "stride3": (1, 1),
        "dense_units": 64,
    },
]

if len(X_train) > 0:
    strat_labels = np.argmax(y_train, axis=1)
    skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    cv_results = []

    for params in param_grid:
        fold_scores = []
        for tr_idx, val_idx in skf.split(X_train, strat_labels):
            X_tr, X_val = X_train[tr_idx], X_train[val_idx]
            y_tr, y_val = y_train[tr_idx], y_train[val_idx]

            model = build_model(params, shape=X_train.shape[1:], classes=y_train.shape[1])

            early_stopping = EarlyStopping(
                monitor='val_top3_acc',
                patience=8,
                mode='max',
                restore_best_weights=True
            )

            model.fit(
                X_tr,
                y_tr,
                epochs=60,
                batch_size=64,
                validation_data=(X_val, y_val),
                callbacks=[early_stopping],
                verbose=0
            )

            y_pred = model.predict(X_val, verbose=0)
            top3 = np.argsort(y_pred, axis=1)[:, -3:]
            true_idx = np.argmax(y_val, axis=1)
            top3_acc = np.mean([t in top3[i] for i, t in enumerate(true_idx)])
            fold_scores.append(top3_acc)

        mean_top3 = float(np.mean(fold_scores))
        cv_results.append((mean_top3, params))
        print(f"Params {params} -> mean top-3 acc: {mean_top3:.4f}")

    best_top3, best_params = max(cv_results, key=lambda x: x[0])
    print(f"Best params: {best_params}")
    print(f"Best mean top-3 acc: {best_top3:.4f}")
else:
    best_params = None



In [ ]:
input_shape = X_train.shape[1:] if len(X_train) else input_shape
num_classes = y_train.shape[1] if len(X_train) else num_classes

default_params = {
    "filters": 32,
    "dense_units": 64,
    "kernel1": (3, 10),
    "stride1": (1, 2),
    "kernel2": (1, 10),
    "stride2": (1, 2),
    "kernel3": (1, 5),
    "stride3": (1, 1),
}

if "best_params" in globals() and best_params:
    model = build_model(best_params, input_shape, num_classes)
else:
    model = build_model(default_params, input_shape, num_classes)

model.summary()



In [ ]:
# Early stopping callback (validation-based)
early_stopping = EarlyStopping(
    monitor='val_top3_acc',
    mode='max',
    patience=20,
    restore_best_weights=True
)

# Model training
history = model.fit(
    X_train,
    y_train,
    epochs=250,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stopping],
    shuffle=True
)



In [ ]:
model.save('cnn_location.keras')


In [ ]:
model = load_model("cnn_location.keras")


In [ ]:
# Impact location labels for decoding confusion matrix axes
IMPACT_LOCATIONS = [
    'Back', 'Back Left', 'Back Neck', 'Back Right', 'Back Top Left', 'Back Top Right',
    'Bottom Back', 'Bottom Back Left', 'Bottom Back Right', 'Bottom Front', 'Bottom Left',
    'Bottom Right', 'Front', 'Front Bottom Left', 'Front Bottom Right', 'Front Left',
    'Front Neck', 'Front Right', 'Front Top Left', 'Front Top Right', 'Left',
    'Left Neck', 'Right', 'Right Neck', 'Top Back', 'Top Front', 'Top Left',
    'Top Right'
]
# Evaluation metrics and confusion matrix
if len(X_test) > 0:
    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)
    y_true = np.argmax(y_test, axis=1)

    print(classification_report(y_true, y_pred, digits=4, target_names=IMPACT_LOCATIONS))

    overall_f1 = f1_score(y_true, y_pred, average='weighted')
    print(f"Overall F1 (weighted): {overall_f1:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    cm = cm.astype(float)
    row_sums = cm.sum(axis=1, keepdims=True)
    cm_norm = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums!=0)
    plt.figure(figsize=(8, 6))
    labels = IMPACT_LOCATIONS if 'IMPACT_LOCATIONS' in globals() else [str(i) for i in range(cm.shape[0])]
    sns.heatmap(cm_norm, annot=False, cmap='Greens', xticklabels=labels, yticklabels=labels, vmin=0, vmax=1)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.show()

    top3 = np.argsort(y_prob, axis=1)[:, -3:]
    top3_acc = np.mean([t in top3[i] for i, t in enumerate(y_true)])
    print(f"Top-3 accuracy: {top3_acc:.4f}")



In [ ]:
# Decode one-hot impact_location and show top-3 predictions per category
IMPACT_LOCATIONS = [
    'Back', 'Back Left', 'Back Neck', 'Back Right', 'Back Top Left', 'Back Top Right',
    'Bottom Back', 'Bottom Back Left', 'Bottom Back Right', 'Bottom Front', 'Bottom Left',
    'Bottom Right', 'Front', 'Front Bottom Left', 'Front Bottom Right', 'Front Left',
    'Front Neck', 'Front Right', 'Front Top Left', 'Front Top Right', 'Left',
    'Left Neck', 'Right', 'Right Neck', 'Top Back', 'Top Front', 'Top Left',
    'Top Right'
]

if len(X_test) > 0:
    y_prob = model.predict(X_test, verbose=0)
    y_true_idx = np.argmax(y_test, axis=1)

    seen = set()
    for i, true_idx in enumerate(y_true_idx):
        if true_idx in seen:
            continue
        seen.add(true_idx)

        true_label = IMPACT_LOCATIONS[true_idx]
        order = np.argsort(y_prob[i])[::-1]
        top3 = []
        for j in order[:3]:
            top3.append((IMPACT_LOCATIONS[j], float(y_prob[i, j])))

        print(f"Category: {true_label}")
        print("Top-3:", top3)

    missing = [IMPACT_LOCATIONS[i] for i in range(len(IMPACT_LOCATIONS)) if i not in seen]
    if missing:
        print("No samples found for:", missing)